# 01. Shape Feature Validation

Purpose: validate the D1 candle-shape feature contract on one domestic
index and one overseas index before any tokenizer is fit.

Reusable feature summary code lives in `kairos.experiments.shape_tokenizer.feature_validation`.
Broker access is isolated in `kairos.sources.brokers`.


## Validation Gates

- Reuse `kairos.core.features.extract_features`.
- Count zero-range candles separately; they must not enter tokenizer
  fitting.
- Keep boundary policy as winsorize + flags and record exclusion loss.
- Keep raw OHLCV out of artifacts.


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from kairos.sources.brokers import fetch_index_candles
from kairos.experiments.artifacts import config_hash, find_project_root, strip_hash_excluded, to_jsonable, write_csv, write_json
from kairos.experiments.shape_tokenizer.feature_validation import (
    D1_FEATURE_REQUESTS,
    KIS_MAJOR_INDEX_DAILY_MAX_PAGES,
    build_feature_validation_config,
    summarize_empty_dataset,
    summarize_dataset,
)
from kairos.experiments.protocol import PHASE_01_ID, RESEARCH_NAME, STEP_01_FEATURE_ID

PROJECT_ROOT = find_project_root()
RUNS_ROOT = PROJECT_ROOT / "notebooks" / "runs" / RESEARCH_NAME
SOURCE_NOTEBOOK = "notebooks/candlestick-shape-quantization/01_shape_feature_validation.ipynb"
SEED = 7

PROJECT_ROOT


PosixPath('/Users/choeseoggyu/portfolio/kairos')

## Dataset Requests


In [2]:
[to_jsonable(request) for request in D1_FEATURE_REQUESTS]


[{'dataset_id': 'd1_kospi_daily',
  'stage': 'D1',
  'symbol': 'KOSPI',
  'market': 'KRX-INDEX',
  'interval': '1d',
  'provider': 'kiwoom',
  'broker_method': 'client.domestic.chart.industry_daily',
  'broker_symbol': '001',
  'request': {'base_date': '2026-07-03',
   'start_date': '1990-01-01',
   'max_pages': None}},
 {'dataset_id': 'd1_kospi_1m',
  'stage': 'D1',
  'symbol': 'KOSPI',
  'market': 'KRX-INDEX',
  'interval': '1m',
  'provider': 'kiwoom',
  'broker_method': 'client.domestic.chart.industry_minute',
  'broker_symbol': '001',
  'request': {'interval_minutes': 1,
   'base_date': '2026-07-03',
   'start_date': '1990-01-01 000000',
   'max_pages': None}},
 {'dataset_id': 'd1_kosdaq_daily',
  'stage': 'D1',
  'symbol': 'KOSDAQ',
  'market': 'KRX-INDEX',
  'interval': '1d',
  'provider': 'kiwoom',
  'broker_method': 'client.domestic.chart.industry_daily',
  'broker_symbol': '101',
  'request': {'base_date': '2026-07-03',
   'start_date': '1990-01-01',
   'max_pages': None}},
 

## Artifact Writers


In [3]:
def make_figure(sample_rows: list[dict[str, Any]], figure_dir: Path, dataset_id: str) -> list[str]:
    figure_dir.mkdir(parents=True, exist_ok=True)
    lambda_path = figure_dir / "lambda_scatter.png"
    hist_path = figure_dir / "shape_core_hist.png"

    lambda_o = [float(row["lambda_o"]) for row in sample_rows]
    lambda_c = [float(row["lambda_c"]) for row in sample_rows]
    s1 = [float(row["s1"]) for row in sample_rows if not row["is_zero_range"]]
    s2 = [float(row["s2"]) for row in sample_rows if not row["is_zero_range"]]

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(lambda_o, lambda_c, s=28, alpha=0.75)
    ax.plot([0, 1], [0, 1], color="black", linewidth=1, alpha=0.4)
    ax.set_xlim(-0.03, 1.03)
    ax.set_ylim(-0.03, 1.03)
    ax.set_xlabel("lambda_o")
    ax.set_ylabel("lambda_c")
    ax.set_title(f"{dataset_id} lambda space")
    fig.tight_layout()
    fig.savefig(lambda_path, dpi=160)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(s1, bins=100, alpha=0.65, label="s1")
    ax.hist(s2, bins=100, alpha=0.65, label="s2")
    ax.set_xlabel("logit(lambda)")
    ax.set_ylabel("count")
    ax.set_title(f"{dataset_id} shape core distribution")
    ax.legend()
    fig.tight_layout()
    fig.savefig(hist_path, dpi=160)
    plt.close(fig)

    return [lambda_path.name, hist_path.name]


def write_run_artifacts(request, metrics, sample_rows, summary_rows) -> dict[str, Any]:
    config = build_feature_validation_config(request)
    cfg_hash = config_hash(config)
    started_at = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    cfg_dir = RUNS_ROOT / PHASE_01_ID / STEP_01_FEATURE_ID / request.dataset_id / f"cfg-{cfg_hash}"
    run_dir = cfg_dir / f"run-{started_at}_seed-{SEED}"
    tables_dir = run_dir / "tables"
    figures_dir = run_dir / "figures"
    tables_dir.mkdir(parents=True, exist_ok=False)

    status = metrics.get("status", "ok")
    skip_reason = metrics.get("skip_reason")
    figure_files = make_figure(sample_rows, figures_dir, request.dataset_id) if sample_rows else []
    write_json(cfg_dir / "config.json", strip_hash_excluded(to_jsonable(config)))
    write_json(run_dir / "experiment_config.json", config | {"seed": SEED, "started_at": started_at})
    write_json(run_dir / "metrics.json", metrics)
    write_csv(tables_dir / "shape_sample.csv", sample_rows)
    write_csv(tables_dir / "shape_summary.csv", summary_rows)

    manifest = {
        "research": RESEARCH_NAME,
        "phase": PHASE_01_ID,
        "step": STEP_01_FEATURE_ID,
        "dataset_id": request.dataset_id,
        "dataset_stage": request.stage,
        "cfg_hash": cfg_hash,
        "seed": SEED,
        "created_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        "source_notebook": SOURCE_NOTEBOOK,
        "broker_source": request.provider,
        "broker_method": request.broker_method,
        "broker_symbol": request.broker_symbol,
        "credential_env": ["KIWOOM_APP_KEY", "KIWOOM_SECRET_KEY"]
        if request.provider == "kiwoom"
        else ["KIS_APP_KEY", "KIS_APP_SECRET_KEY"],
        "raw_ohlcv_persisted": False,
        "requested_start_date": request.request.get("start_date") or request.request.get("start"),
        "requested_end_date": request.request.get("base_date") or request.request.get("end"),
        "requested_max_pages": request.request.get("max_pages"),
        "pagination_policy": (
            "KIS major-index daily pagination uses FID_INPUT_DATE_1 as the fixed start "
            "and moves FID_INPUT_DATE_2 backward to the day before the oldest returned row."
            if request.broker_method == "client.overseas.chart.major_index"
            else None
        ),
        "status": status,
        "skip_reason": skip_reason,
        "output_files": [
            "experiment_config.json",
            "metrics.json",
            "manifest.json",
            "tables/shape_sample.csv",
            "tables/shape_summary.csv",
            *[f"figures/{name}" for name in figure_files],
        ],
        "notes": [
            "Feature validation only; raw OHLCV rows were not persisted.",
            "Shape math is imported from kairos.core.features via kairos.experiments.shape_tokenizer.feature_validation.",
            "KIS overseas major-index daily history is paged by date-window requests, not tr_cont.",
        ],
    }
    write_json(run_dir / "manifest.json", manifest)
    return {
        "dataset_id": request.dataset_id,
        "cfg_hash": cfg_hash,
        "run_dir": run_dir.as_posix(),
        "status": status,
        "skip_reason": skip_reason,
        "row_count": metrics["row_count"],
        "date_range": metrics["date_range"],
        "provider": request.provider,
        "broker_method": request.broker_method,
        "broker_symbol": request.broker_symbol,
        "requested_start_date": metrics["data_request"]["requested_start_date"],
        "requested_end_date": metrics["data_request"]["requested_end_date"],
        "requested_max_pages": metrics["data_request"]["requested_max_pages"],
        "zero_range_count": metrics["exceptional_rows"]["zero_range_count"],
        "boundary_count": metrics["exceptional_rows"]["boundary_count"],
        "figure_files": figure_files,
    }


## Execute D1 Validation


In [4]:
async def run_feature_validation() -> list[dict[str, Any]]:
    results: list[dict[str, Any]] = []
    for request in D1_FEATURE_REQUESTS:
        candles = await fetch_index_candles(request, env_path=PROJECT_ROOT / ".env")
        if candles:
            metrics, sample_rows, summary_rows = summarize_dataset(request, candles)
        else:
            metrics, sample_rows, summary_rows = summarize_empty_dataset(
                request,
                reason=f"{request.dataset_id} returned no rows from {request.provider} {request.broker_method}",
            )
        results.append(write_run_artifacts(request, metrics, sample_rows, summary_rows))
    return results


RUN_RESULTS = await run_feature_validation()
RUN_RESULTS


[{'dataset_id': 'd1_kospi_daily',
  'cfg_hash': '5628106a',
  'run_dir': '/Users/choeseoggyu/portfolio/kairos/notebooks/runs/candlestick-shape-quantization/phase-01-shape-tokenizer/step-01-shape-feature-validation/d1_kospi_daily/cfg-5628106a/run-20260706-134241_seed-7',
  'status': 'ok',
  'skip_reason': None,
  'row_count': 9420,
  'date_range': {'start': '1990-01-03', 'end': '2026-07-03'},
  'provider': 'kiwoom',
  'broker_method': 'client.domestic.chart.industry_daily',
  'broker_symbol': '001',
  'requested_start_date': '1990-01-01',
  'requested_end_date': '2026-07-03',
  'requested_max_pages': None,
  'zero_range_count': 0,
  'boundary_count': 2582,
  'figure_files': ['lambda_scatter.png', 'shape_core_hist.png']},
 {'dataset_id': 'd1_kospi_1m',
  'cfg_hash': 'd7a389ba',
  'run_dir': '/Users/choeseoggyu/portfolio/kairos/notebooks/runs/candlestick-shape-quantization/phase-01-shape-tokenizer/step-01-shape-feature-validation/d1_kospi_1m/cfg-d7a389ba/run-20260706-134257_seed-7',
  'st

## Acceptance Checks


In [5]:
assert len(RUN_RESULTS) == len(D1_FEATURE_REQUESTS)
assert all(result["status"] in {"ok", "skipped"} for result in RUN_RESULTS)
assert any(result["status"] == "ok" and result["row_count"] > 0 for result in RUN_RESULTS)
assert all(Path(result["run_dir"], "metrics.json").exists() for result in RUN_RESULTS)
assert all(Path(result["run_dir"], "tables", "shape_sample.csv").exists() for result in RUN_RESULTS)
assert all(
    Path(result["run_dir"], "figures", "lambda_scatter.png").exists()
    for result in RUN_RESULTS
    if result["status"] == "ok"
)

kis_daily_requests = [
    request
    for request in D1_FEATURE_REQUESTS
    if request.broker_method == "client.overseas.chart.major_index"
]
assert kis_daily_requests
assert all(
    request.request["max_pages"] == KIS_MAJOR_INDEX_DAILY_MAX_PAGES
    for request in kis_daily_requests
)
assert all("end" in request.request for request in kis_daily_requests)

RUN_RESULTS


[{'dataset_id': 'd1_kospi_daily',
  'cfg_hash': '5628106a',
  'run_dir': '/Users/choeseoggyu/portfolio/kairos/notebooks/runs/candlestick-shape-quantization/phase-01-shape-tokenizer/step-01-shape-feature-validation/d1_kospi_daily/cfg-5628106a/run-20260706-134241_seed-7',
  'status': 'ok',
  'skip_reason': None,
  'row_count': 9420,
  'date_range': {'start': '1990-01-03', 'end': '2026-07-03'},
  'provider': 'kiwoom',
  'broker_method': 'client.domestic.chart.industry_daily',
  'broker_symbol': '001',
  'requested_start_date': '1990-01-01',
  'requested_end_date': '2026-07-03',
  'requested_max_pages': None,
  'zero_range_count': 0,
  'boundary_count': 2582,
  'figure_files': ['lambda_scatter.png', 'shape_core_hist.png']},
 {'dataset_id': 'd1_kospi_1m',
  'cfg_hash': 'd7a389ba',
  'run_dir': '/Users/choeseoggyu/portfolio/kairos/notebooks/runs/candlestick-shape-quantization/phase-01-shape-tokenizer/step-01-shape-feature-validation/d1_kospi_1m/cfg-d7a389ba/run-20260706-134257_seed-7',
  'st

In [6]:
import csv
from math import ceil
from IPython.display import display


def _csv_bool(value: str) -> bool:
    return value.strip().lower() in {"1", "true", "yes"}


def _shape_sample_path(result: dict[str, Any]) -> Path:
    sample_path = Path(result["run_dir"], "tables", "shape_sample.csv")
    if sample_path.exists():
        return sample_path
    cfg_hash = result.get("cfg_hash")
    if not cfg_hash:
        return sample_path
    cfg_dir = (
        RUNS_ROOT
        / PHASE_01_ID
        / STEP_01_FEATURE_ID
        / result["dataset_id"]
        / f"cfg-{cfg_hash}"
    )
    candidates = sorted(cfg_dir.glob("run-*_seed-*/tables/shape_sample.csv"))
    return candidates[-1] if candidates else sample_path


def _shape_core_rows(result: dict[str, Any]) -> list[dict[str, float]]:
    sample_path = _shape_sample_path(result)
    if result["status"] != "ok" or not sample_path.exists():
        return []
    rows: list[dict[str, float]] = []
    with sample_path.open(newline="", encoding="utf-8") as file:
        for row in csv.DictReader(file):
            if _csv_bool(row["is_zero_range"]):
                continue
            rows.append({"s1": float(row["s1"]), "s2": float(row["s2"])})
    return rows


shape_core_by_dataset = {
    result["dataset_id"]: _shape_core_rows(result)
    for result in RUN_RESULTS
}
shape_core_values = [
    value
    for rows in shape_core_by_dataset.values()
    for row in rows
    for value in (row["s1"], row["s2"])
]
assert shape_core_values, "no non-zero-range shape core rows to visualize"

axis_limit = max(2.0, max(abs(value) for value in shape_core_values))
axis_limit = min(axis_limit, 12.0)
cols = 3
rows = ceil(len(RUN_RESULTS) / cols)
fig, axes = plt.subplots(
    rows,
    cols,
    figsize=(4.4 * cols, 3.1 * rows),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
flat_axes = list(axes.ravel()) if hasattr(axes, "ravel") else [axes]
bins = 48
hist_range = (-axis_limit, axis_limit)

for ax, result in zip(flat_axes, RUN_RESULTS, strict=False):
    dataset_id = result["dataset_id"]
    rows_for_dataset = shape_core_by_dataset[dataset_id]
    if rows_for_dataset:
        s1 = [row["s1"] for row in rows_for_dataset]
        s2 = [row["s2"] for row in rows_for_dataset]
        ax.hist(
            s1,
            bins=bins,
            range=hist_range,
            density=True,
            alpha=0.58,
            label="s1",
            color="#2563eb",
        )
        ax.hist(
            s2,
            bins=bins,
            range=hist_range,
            density=True,
            alpha=0.52,
            label="s2",
            color="#dc2626",
        )
        ax.set_title(f"{dataset_id}\nn={len(rows_for_dataset):,}", fontsize=10)
    else:
        ax.text(0.5, 0.5, "skipped / no rows", ha="center", va="center", fontsize=9)
        ax.set_title(f"{dataset_id}\nskipped", fontsize=10)
    ax.axvline(0, color="black", linewidth=0.8, alpha=0.55)
    ax.grid(axis="y", alpha=0.2)

for ax in flat_axes[len(RUN_RESULTS):]:
    ax.axis("off")

for ax in flat_axes[-cols:]:
    ax.set_xlabel("logit(lambda)")
for ax in flat_axes[::cols]:
    ax.set_ylabel("density")

handles, labels = flat_axes[0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="upper right", bbox_to_anchor=(0.985, 0.985))
fig.suptitle("D1 dataset shape core histogram overview", fontsize=14)

overview_path = RUNS_ROOT / PHASE_01_ID / STEP_01_FEATURE_ID / "shape_core_histogram_overview.png"
overview_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(overview_path, dpi=180)
display(fig)
plt.close(fig)

overview_path


<Figure size 1320x1240 with 12 Axes>

PosixPath('/Users/choeseoggyu/portfolio/kairos/notebooks/runs/candlestick-shape-quantization/phase-01-shape-tokenizer/step-01-shape-feature-validation/shape_core_histogram_overview.png')